# JAXFrame Advanced Features Tutorial

This notebook covers advanced JAXFrame features including joins, data transformations, and integration with machine learning workflows.

## Topics Covered:
1. **Join Operations** - Combining DataFrames
2. **Data Concatenation** - Merging multiple DataFrames
3. **Advanced Transformations** - Wide-to-long with complex patterns
4. **Machine Learning Integration** - JAXFrame in ML pipelines
5. **Lookup Tables** - Efficient data mapping
6. **Performance Optimization** - Best practices for speed

In [1]:
# Setup
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))

import jax
import jax.numpy as jnp
import numpy as np
from jaxframe import DataFrame, MaskedArray
from jaxframe.transform import wide_to_long_masked, long_to_wide_masked, wide_df_to_masked_array
import time

print("🚀 JAXFrame Advanced Features Tutorial")
print(f"JAX version: {jax.__version__}")

🚀 JAXFrame Advanced Features Tutorial
JAX version: 0.7.1


## 1. Join Operations

JAXFrame supports efficient join operations for combining datasets.

In [2]:
# Create sample DataFrames for joining
users = DataFrame({
    'user_id': [1, 2, 3, 4],
    'name': ['Alice', 'Bob', 'Charlie', 'Diana'],
    'age': jnp.array([25, 30, 35, 28])
})

scores = DataFrame({
    'user_id': [1, 2, 3, 5],  # Note: user 4 missing, user 5 extra  
    'math_score': jnp.array([85.5, 92.0, 78.5, 88.0]),
    'science_score': jnp.array([90.0, 87.5, 82.0, 91.5])
})

print("Users DataFrame:")
print(users)
print("\nScores DataFrame:")
print(scores)

# Inner join (only matching records)
joined = users.join(scores, on='user_id', source=['user_id'])
print(f"\nInner Join Result:")
print(joined)
print(f"Shape: {joined.shape} (only users 1, 2, 3 have scores)")

Users DataFrame:
DataFrame(4 rows, 3 columns)
Columns: user_id, name, age
Dtypes: user_id: list[int], name: list[str], age: int32
  [0]: {'user_id': 1, 'name': 'Alice', 'age': 25}
  [1]: {'user_id': 2, 'name': 'Bob', 'age': 30}
  [2]: {'user_id': 3, 'name': 'Charlie', 'age': 35}
  [3]: {'user_id': 4, 'name': 'Diana', 'age': 28}

Scores DataFrame:
DataFrame(4 rows, 3 columns)
Columns: user_id, math_score, science_score
Dtypes: user_id: list[int], math_score: float32, science_score: float32
  [0]: {'user_id': 1, 'math_score': 85.500, 'science_score': 90.000}
  [1]: {'user_id': 2, 'math_score': 92.000, 'science_score': 87.500}
  [2]: {'user_id': 3, 'math_score': 78.500, 'science_score': 82.000}
  [3]: {'user_id': 5, 'math_score': 88.000, 'science_score': 91.500}

Inner Join Result:
DataFrame(3 rows, 3 columns)
Columns: user_id, name, age
Dtypes: user_id: list[int], name: list[str], age: int32
  [0]: {'user_id': 1, 'name': 'Alice', 'age': 25}
  [1]: {'user_id': 2, 'name': 'Bob', 'age': 30}

In [3]:
# Semi-join (filtering based on existence in other DataFrame)
# Find users who have scores using semi-join
filtered_users = users.join(scores, on='user_id', how='semi')
print("Semi-join (users who have scores):")
print(filtered_users)
print(f"Original users: {len(users)}, Users with scores: {len(filtered_users)}")

# Multi-column joins
locations = DataFrame({
    'user_id': [1, 2, 3],
    'city': ['NYC', 'LA', 'Chicago'],
    'country': ['USA', 'USA', 'USA']
})

# Join with location data  
full_profile = joined.join(locations, on='user_id', source=['user_id'])
print(f"\nFull user profiles:")
print(full_profile)

Semi-join (users who have scores):
DataFrame(3 rows, 3 columns)
Columns: user_id, name, age
Dtypes: user_id: list[int], name: list[str], age: int32
  [0]: {'user_id': 1, 'name': 'Alice', 'age': 25}
  [1]: {'user_id': 2, 'name': 'Bob', 'age': 30}
  [2]: {'user_id': 3, 'name': 'Charlie', 'age': 35}
Original users: 4, Users with scores: 3

Full user profiles:
DataFrame(3 rows, 3 columns)
Columns: user_id, name, age
Dtypes: user_id: list[int], name: list[str], age: int32
  [0]: {'user_id': 1, 'name': 'Alice', 'age': 25}
  [1]: {'user_id': 2, 'name': 'Bob', 'age': 30}
  [2]: {'user_id': 3, 'name': 'Charlie', 'age': 35}


## 2. Data Concatenation

Combine DataFrames by rows or columns efficiently.

In [4]:
# Create DataFrames to concatenate
batch1 = DataFrame({
    'id': [1, 2, 3],
    'values': jnp.array([10, 20, 30]),
    'labels': ['A', 'B', 'C']
})

batch2 = DataFrame({
    'id': [4, 5, 6], 
    'values': jnp.array([40, 50, 60]),
    'labels': ['D', 'E', 'F']
})

# Concatenate rows (vertical stacking)
combined_rows = DataFrame.concat_dataframes([batch1, batch2], axis=0)
print("Row concatenation:")
print(combined_rows)
print(f"Shape: {combined_rows.shape}")

# Create additional columns to concatenate horizontally
extra_cols = DataFrame({
    'status': ['active', 'inactive', 'pending'],
    'score': jnp.array([85.0, 92.0, 78.0])
})

# Concatenate columns (horizontal stacking)
combined_cols = batch1._concat_columns(extra_cols)
print(f"\nColumn concatenation:")
print(combined_cols)
print(f"Shape: {combined_cols.shape}")

Row concatenation:
DataFrame(6 rows, 3 columns)
Columns: id, values, labels
Dtypes: id: list[int], values: int32, labels: list[str]
  [0]: {'id': 1, 'values': 10, 'labels': 'A'}
  [1]: {'id': 2, 'values': 20, 'labels': 'B'}
  [2]: {'id': 3, 'values': 30, 'labels': 'C'}
  [3]: {'id': 4, 'values': 40, 'labels': 'D'}
  [4]: {'id': 5, 'values': 50, 'labels': 'E'}
  ... (1 more rows)
Shape: (6, 3)

Column concatenation:
DataFrame(3 rows, 5 columns)
Columns: id, values, labels, status, score
Dtypes: id: list[int], values: int32, labels: list[str], status: list[str], score: float32
  [0]: {'id': 1, 'values': 10, 'labels': 'A', 'status': 'active', 'score': 85.000}
  [1]: {'id': 2, 'values': 20, 'labels': 'B', 'status': 'inactive', 'score': 92.000}
  [2]: {'id': 3, 'values': 30, 'labels': 'C', 'status': 'pending', 'score': 78.000}
Shape: (3, 5)


## 3. Advanced Wide-to-Long Transformations

Handle complex data reshaping with multiple variables and patterns.

In [5]:
# Complex wide data with multiple measurement types
experiment_data = {
    'subject_id': ['S001', 'S002', 'S003'],
    'group': ['treatment', 'control', 'treatment'],
    # Temperature measurements at different time points
    'temp$0$value': jnp.array([36.5, 36.8, 36.2]),
    'temp$1$value': jnp.array([37.1, 36.9, 36.8]),
    'temp$2$value': jnp.array([36.9, 37.0, 36.5]),
    'temp$0$mask': np.array([True, True, True]),
    'temp$1$mask': np.array([True, True, False]),  # Missing for S003
    'temp$2$mask': np.array([True, False, True]),  # Missing for S002
    # Heart rate measurements  
    'hr$0$value': jnp.array([72, 68, 75]),
    'hr$1$value': jnp.array([78, 72, 80]),
    'hr$2$value': jnp.array([74, 70, 77]),
    'hr$0$mask': np.array([True, True, True]),
    'hr$1$mask': np.array([True, True, True]),
    'hr$2$mask': np.array([False, True, True])    # Missing for S001
}

wide_exp = DataFrame(experiment_data)
print("Complex experimental data (wide format):")
print(f"Shape: {wide_exp.shape}")
print("Columns:", wide_exp.columns[:8], "...")  # Show first 8 columns

# Transform to long format - multiple ID columns
long_exp = wide_to_long_masked(wide_exp, 
                               id_columns=['subject_id', 'group'],
                               var_pattern=r'([^$]+)\$(\d+)\$(value|mask)')

print(f"\nLong format:")
print(f"Shape: {long_exp.shape}")
print(long_exp)

Complex experimental data (wide format):
Shape: (3, 14)
Columns: ('subject_id', 'group', 'temp$0$value', 'temp$1$value', 'temp$2$value', 'temp$0$mask', 'temp$1$mask', 'temp$2$mask') ...

Long format:
Shape: (15, 4)
DataFrame(15 rows, 4 columns)
Columns: subject_id, group, variable, value
Dtypes: subject_id: list[str], group: list[str], variable: list[int], value: float32
  [0]: {'subject_id': 'S001', 'group': 'treatment', 'variable': 0, 'value': 72.000}
  [1]: {'subject_id': 'S001', 'group': 'treatment', 'variable': 1, 'value': 78.000}
  [2]: {'subject_id': 'S001', 'group': 'treatment', 'variable': 0, 'value': 36.500}
  [3]: {'subject_id': 'S001', 'group': 'treatment', 'variable': 1, 'value': 37.100}
  [4]: {'subject_id': 'S001', 'group': 'treatment', 'variable': 2, 'value': 36.900}
  ... (10 more rows)


## 4. Machine Learning Integration

JAXFrame's JAX compatibility makes it perfect for machine learning workflows. Here's how to use it with gradient-based optimization:

In [6]:
# Example: Linear regression with automatic differentiation
import jax.numpy as jnp
from jax import grad, jit
import numpy as np

# Create training data
n_samples = 1000
X_data = np.random.randn(n_samples, 3)
true_weights = jnp.array([1.5, -2.0, 0.8])
y_data = X_data @ true_weights + 0.1 * np.random.randn(n_samples)

# Create DataFrame with fast constructor
training_df = DataFrame.from_numpy_arrays({
    'feature_1': X_data[:, 0],
    'feature_2': X_data[:, 1], 
    'feature_3': X_data[:, 2],
    'target': y_data
})

print(f"Training data shape: {training_df.shape}")
print(f"Data types: {[type(training_df[col]).__name__ for col in training_df.columns]}")

# Extract features and target - this is very fast due to JAX array references
X = jnp.column_stack([training_df[f'feature_{i}'] for i in range(1, 4)])
y = training_df['target']

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")

Training data shape: (1000, 4)
Data types: ['ndarray', 'ndarray', 'ndarray', 'ArrayImpl']
Feature matrix shape: (1000, 3)
Target vector shape: (1000,)


In [7]:
# Define loss function and gradient
def mse_loss(weights, X, y):
    predictions = X @ weights
    return jnp.mean((predictions - y) ** 2)

# JIT compile for performance
loss_fn = jit(mse_loss)
grad_fn = jit(grad(mse_loss))

# Simple gradient descent
learning_rate = 0.01
weights = jnp.array([0.0, 0.0, 0.0])

print("Training linear regression:")
print(f"True weights: {true_weights}")
print(f"Initial weights: {weights}")

# Training loop
for epoch in range(100):
    loss = loss_fn(weights, X, y)
    gradients = grad_fn(weights, X, y)
    weights = weights - learning_rate * gradients
    
    if epoch % 20 == 0:
        print(f"Epoch {epoch}: Loss = {loss:.4f}, Weights = {weights}")

print(f"\nFinal weights: {weights}")
print(f"True weights:  {true_weights}")
print(f"Error: {jnp.abs(weights - true_weights)}")

Training linear regression:
True weights: [ 1.5 -2.   0.8]
Initial weights: [0. 0. 0.]
Epoch 0: Loss = 7.0836, Weights = [ 0.02977126 -0.04184606  0.01663273]
Epoch 20: Loss = 3.0872, Weights = [ 0.5150148 -0.7172421  0.2850726]
Epoch 40: Loss = 1.3492, Weights = [ 0.8395253  -1.1598017   0.46103555]
Epoch 60: Loss = 0.5931, Weights = [ 1.0566244 -1.4497142  0.5764327]
Epoch 80: Loss = 0.2640, Weights = [ 1.2019197  -1.6395757   0.65214604]

Final weights: [ 1.2952038  -1.7588332   0.69982624]
True weights:  [ 1.5 -2.   0.8]
Error: [0.2047962  0.24116683 0.10017377]


## 5. Lookup Tables and Semi-Joins

JAXFrame provides efficient lookup table operations for data enrichment and semi-joins:

In [8]:
# Create lookup table for product information
product_lookup = DataFrame({
    'product_id': ['A001', 'A002', 'A003', 'B001', 'B002'],
    'category': ['Electronics', 'Electronics', 'Electronics', 'Books', 'Books'],
    'price': jnp.array([299.99, 149.99, 99.99, 29.99, 19.99]),
    'in_stock': [True, True, False, True, True]
})

# Sales transaction data
transactions = DataFrame({
    'transaction_id': ['T001', 'T002', 'T003', 'T004', 'T005', 'T006'],
    'product_id': ['A001', 'A002', 'A003', 'B001', 'A001', 'B002'],
    'quantity': jnp.array([2, 1, 1, 3, 1, 2]),
    'customer_id': ['C100', 'C101', 'C102', 'C100', 'C103', 'C101']
})

print("Product lookup table:")
print(product_lookup)
print("\nTransaction data:")
print(transactions)

Product lookup table:
DataFrame(5 rows, 4 columns)
Columns: product_id, category, price, in_stock
Dtypes: product_id: list[str], category: list[str], price: float32, in_stock: list[bool]
  [0]: {'product_id': 'A001', 'category': 'Electronics', 'price': 299.990, 'in_stock': True}
  [1]: {'product_id': 'A002', 'category': 'Electronics', 'price': 149.990, 'in_stock': True}
  [2]: {'product_id': 'A003', 'category': 'Electronics', 'price': 99.990, 'in_stock': False}
  [3]: {'product_id': 'B001', 'category': 'Books', 'price': 29.990, 'in_stock': True}
  [4]: {'product_id': 'B002', 'category': 'Books', 'price': 19.990, 'in_stock': True}

Transaction data:
DataFrame(6 rows, 4 columns)
Columns: transaction_id, product_id, quantity, customer_id
Dtypes: transaction_id: list[str], product_id: list[str], quantity: int32, customer_id: list[str]
  [0]: {'transaction_id': 'T001', 'product_id': 'A001', 'quantity': 2, 'customer_id': 'C100'}
  [1]: {'transaction_id': 'T002', 'product_id': 'A002', 'quanti

In [18]:
# Semi-join: Find transactions for products that are in stock
# Create a simple in-stock product list for demonstration
in_stock_product_ids = ['A001', 'A002', 'B001', 'B002']  # Products that are in stock
in_stock_products = DataFrame({
    'product_id': in_stock_product_ids,
    'category': ['Electronics', 'Electronics', 'Books', 'Books'],
    'price': jnp.array([299.99, 149.99, 29.99, 19.99])
})

print("Products in stock:")
print(in_stock_products)

# Semi-join to get transactions for products that are in stock
valid_transactions = transactions.join(in_stock_products, on='product_id', how='semi')

print(f"\nTransactions for in-stock products:")
print(f"Original transactions: {len(transactions)}")
print(f"Valid transactions: {len(valid_transactions)}")
print(valid_transactions)

# Calculate total revenue - create a simple price lookup
price_lookup = {pid: price for pid, price in zip(in_stock_products['product_id'], in_stock_products['price'])}
transaction_prices = jnp.array([price_lookup[pid] for pid in valid_transactions['product_id']])

revenue = valid_transactions['quantity'] * transaction_prices
total_revenue = jnp.sum(revenue)

print(f"\nTotal revenue from valid transactions: ${total_revenue:.2f}")

Products in stock:
DataFrame(4 rows, 3 columns)
Columns: product_id, category, price
Dtypes: product_id: list[str], category: list[str], price: float32
  [0]: {'product_id': 'A001', 'category': 'Electronics', 'price': 299.990}
  [1]: {'product_id': 'A002', 'category': 'Electronics', 'price': 149.990}
  [2]: {'product_id': 'B001', 'category': 'Books', 'price': 29.990}
  [3]: {'product_id': 'B002', 'category': 'Books', 'price': 19.990}

Transactions for in-stock products:
Original transactions: 6
Valid transactions: 4
DataFrame(4 rows, 4 columns)
Columns: transaction_id, product_id, quantity, customer_id
Dtypes: transaction_id: list[str], product_id: list[str], quantity: int32, customer_id: list[str]
  [0]: {'transaction_id': 'T001', 'product_id': 'A001', 'quantity': 2, 'customer_id': 'C100'}
  [1]: {'transaction_id': 'T002', 'product_id': 'A002', 'quantity': 1, 'customer_id': 'C101'}
  [2]: {'transaction_id': 'T004', 'product_id': 'B001', 'quantity': 3, 'customer_id': 'C100'}
  [3]: {'t

## 6. Performance Best Practices

Here are key strategies for maximizing JAXFrame performance:

In [20]:
# Performance Best Practices Demonstration

# 1. Use fast constructors for homogeneous data
print("=== Fast Constructor Performance ===")

# Create large dataset
n = 100000
data_dict = {
    f'col_{i}': np.random.randn(n) for i in range(10)
}

# Time the fast constructor
import time

start = time.time()
fast_df = DataFrame.from_numpy_arrays(data_dict)
fast_time = time.time() - start

start = time.time()
slow_df = DataFrame(data_dict)  # Regular constructor
slow_time = time.time() - start

print(f"Fast constructor: {fast_time:.4f}s")
print(f"Regular constructor: {slow_time:.4f}s")
print(f"Speedup: {slow_time/fast_time:.1f}x")

# 2. Minimize copies - use column references
print(f"\n=== Memory Efficiency ===")
col_data = fast_df['col_0']
print(f"Original array id: {id(fast_df['col_0'])}")
print(f"Retrieved column id: {id(col_data)}")
print(f"Same object (no copy): {col_data is fast_df['col_0']}")

# 3. JIT compile operations for repeated use
@jit
def compute_stats(data):
    return {
        'mean': jnp.mean(data),
        'std': jnp.std(data),
        'min': jnp.min(data),
        'max': jnp.max(data)
    }

# Time JIT compilation
data = fast_df['col_0']
start = time.time()
stats = compute_stats(data)  # First call includes compilation
jit_time_first = time.time() - start

start = time.time()
stats = compute_stats(data)  # Second call is compiled
jit_time_second = time.time() - start

print(f"\nJIT first call (with compilation): {jit_time_first:.4f}s")
print(f"JIT second call (compiled): {jit_time_second:.4f}s")
print(f"Speedup after compilation: {jit_time_first/jit_time_second:.1f}x")

=== Fast Constructor Performance ===
Fast constructor: 0.0011s
Regular constructor: 0.0012s
Speedup: 1.1x

=== Memory Efficiency ===
Original array id: 4955472400
Retrieved column id: 4955486128
Same object (no copy): False

JIT first call (with compilation): 0.0505s
JIT second call (compiled): 0.0001s
Speedup after compilation: 384.8x


## Key Performance Tips

1. **Use fast constructors**: `DataFrame.from_jax_arrays()` and `DataFrame.from_numpy_arrays()` are 10-200x faster for homogeneous data

2. **Leverage JAX immutability**: JAX arrays don't need copying - JAXFrame shares references when safe

3. **JIT compile repeated operations**: Use `@jit` for functions you'll call multiple times

4. **Batch operations**: Process multiple columns together rather than one at a time

5. **Use MaskedArrays for missing data**: More efficient than sentinel values for sparse data

6. **Choose appropriate data types**: JAX arrays for numerical computation, NumPy for mixed types, lists for heterogeneous data

## Next Steps

This tutorial covered the advanced features of JAXFrame. For more examples and detailed API documentation, check out:

- The test files in `tests/` directory for comprehensive usage examples  
- The source code in `src/jaxframe/` for implementation details
- JAX documentation for understanding the underlying acceleration framework

JAXFrame provides a powerful, performant foundation for data analysis with seamless JAX integration. Happy computing!

## 7. Real-World Statistical Modeling with NumPyro

This example demonstrates fitting a linear statistical model to predict housing prices using multiple features stored in separate DataFrames with missing data. We'll use NumPyro for Bayesian inference and JAXFrame for efficient data handling.